# E4 -- two-component coupled quartic chain (24D): run notebook

**This notebook runs and saves. It does not typeset figures.**

Every official metric is computed here, at run time, and written into each run's `metrics_timeseries.csv` and `cost_timeseries.csv`. The companion notebook `E4_coupled_quartic_chain_plot.ipynb` reads those numbers and never recomputes them.

**Run All executes the single default full configuration.** There is exactly one configuration for this experiment, `configs/experiments/E4.yaml` -- there is no smoke, dev, reduced, or production profile to choose between. Lowering the particle count for local debugging is an explicit temporary edit, never a second committed profile.

Each variant is saved the moment it finishes, into its own atomically renamed run directory, so a variant that fails leaves the earlier ones untouched.

In [1]:
import sys

sys.path.insert(0, "..")  # importable when launched from notebooks/

from src.pipeline import load_experiment, run_variants_and_save

## Target, reference, and cost calibration

The reference is built **once** and reused by every method. It does not depend on any method parameter, so it is **never rebuilt per method, per hyperparameter value, or per canonical/tamed variant**; a cached reference on disk is loaded instead of being recomputed.

The force-equivalent-evaluation (FEE) calibration is measured once per device in the same way, and every run in this experiment is costed against that one calibration. The device is resolved automatically -- no device index is pinned in this notebook.

In [2]:
experiment = load_experiment("E4", device="auto")

reference = experiment.ensure_reference()
fee = experiment.ensure_fee_calibration()

described = reference.describe()
print(f"reference: kind={described.get('kind', described.get('method'))}  hash={experiment.reference_hash}")
print(f"FEE:       unit={fee.cost_unit}  hash={fee.hash}")

reference: kind=multistart_pt_mala_with_snis_crosscheck  hash=a7c5eaa23f080ec8d57b760aa0223d05
FEE:       unit=amortized_time_per_configuration  hash=617414aeae69572deaa374dae2d362a1


## ULA

Every taming-capable method runs **both** a canonical and a tamed variant. `run_variants_and_save` expands each entry of `variants` into those two runs by itself, so a notebook never passes `tame`. The two variants are **calibrated separately** -- each one gets its own step size from its own `dt` refinement -- and each is saved as its own run directory.

In [3]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="ULA",
    # `run_variants_and_save` expands each entry below into a
    # canonical and a tamed run, so `tame` is never passed here.
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[ULA, canonical] saved to /home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/ULA/ULA-canonical-dt0.002-20260806T221607062676Z


[ULA, tamed] saved to /home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/ULA/ULA-tamed-dt0.002-20260806T221622728509Z


[{'variant_label': 'ULA, canonical',
  'status': 'complete',
  'run_id': 'ULA-canonical-dt0.002-20260806T221607062676Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/ULA/ULA-canonical-dt0.002-20260806T221607062676Z',
  'dt': 0.002,
  'calibration_hash': 'a18785fcb143b8fd1194b6bfff1bea90',
  'fee_calibration_hash': '617414aeae69572deaa374dae2d362a1',
  'n_metric_rows': 884,
  'n_snapshots': 4},
 {'variant_label': 'ULA, tamed',
  'status': 'complete',
  'run_id': 'ULA-tamed-dt0.002-20260806T221622728509Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/ULA/ULA-tamed-dt0.002-20260806T221622728509Z',
  'dt': 0.002,
  'calibration_hash': '56134e61b2259b61d2cc547df37276ca',
  'fee_calibration_hash': '617414aeae69572deaa374dae2d362a1',
  'n_metric_rows': 884,
  'n_snapshots': 4}]

## MALA

MALA supports taming, so it also runs both variants. Tamed MALA implements the actual tamed proposal density in the Metropolis-Hastings ratio; it is a genuine second sampler, not a relabelled copy of the canonical run.

In [4]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="MALA",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[MALA, canonical] NOT CALIBRATABLE: unstable at every timestep tried (mh_acceptance_outside_target_band, temporal_ess_fraction) though it improves as the timestep shrinks


[MALA, tamed] NOT CALIBRATABLE: unstable at every timestep tried (mh_acceptance_outside_target_band, temporal_ess_fraction) though it improves as the timestep shrinks


[{'variant_label': 'MALA, canonical',
  'method': 'MALA',
  'status': 'uncalibratable',
  'diagnosis': 'unstable at every timestep tried (mh_acceptance_outside_target_band, temporal_ess_fraction) though it improves as the timestep shrinks',
  'calibration_kind': 'timestep',
  'calibration_table': [{'dt': 0.004,
    'pass': False,
    'stability_problems': [('temporal_ess_fraction', 0.004417418362227037),
     ('mh_acceptance_outside_target_band', 0.7521156770166453)],
    'agreement_failures': [],
    'summary': {'n_steps': 1562,
     'nonfinite_fraction': 0.0,
     'boundary_reject_fraction': 0.0,
     'temporal_ess': 6.900007481798632,
     'temporal_ess_fraction': 0.004417418362227037,
     'temporal_ess_draws_per_seed': 781,
     'n_effective': 1024,
     'summary_mean': -0.9715781255757082,
     'summary_mean_se': 0.00425687230494431,
     'summary_abs_mean': 0.9715781255757082,
     'summary_abs_mean_se': 0.00425687230494431,
     'summary_median': -0.9790584790037291,
     'summ

## FLA

The three stability indices are this experiment's default grid in `configs/registry.yaml`. All three run from this one cell and save as separate variants, and each of them is expanded into a canonical and a tamed run.

In [5]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="FLA",
    variants=[
        {"alpha": 1.6}, {"alpha": 1.7}, {"alpha": 1.8},
    ],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[FLA alpha=1.6, canonical] saved to /home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/FLA/FLA-alpha1.6-canonical-dt0.001-20260806T222326433619Z


[FLA alpha=1.6, tamed] saved to /home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/FLA/FLA-alpha1.6-tamed-dt0.0005-20260806T222424452663Z


[FLA alpha=1.7, canonical] saved to /home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/FLA/FLA-alpha1.7-canonical-dt0.0005-20260806T222514042225Z


[FLA alpha=1.7, tamed] saved to /home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/FLA/FLA-alpha1.7-tamed-dt0.0005-20260806T222608398207Z


[FLA alpha=1.8, canonical] saved to /home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/FLA/FLA-alpha1.8-canonical-dt0.0005-20260806T222658651665Z


[FLA alpha=1.8, tamed] saved to /home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/FLA/FLA-alpha1.8-tamed-dt0.0005-20260806T222753102296Z


[{'variant_label': 'FLA alpha=1.6, canonical',
  'status': 'complete',
  'run_id': 'FLA-alpha1.6-canonical-dt0.001-20260806T222326433619Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/FLA/FLA-alpha1.6-canonical-dt0.001-20260806T222326433619Z',
  'dt': 0.001,
  'calibration_hash': '21e600737f38c81ea2322726a72c09da',
  'fee_calibration_hash': '617414aeae69572deaa374dae2d362a1',
  'n_metric_rows': 884,
  'n_snapshots': 4},
 {'variant_label': 'FLA alpha=1.6, tamed',
  'status': 'complete',
  'run_id': 'FLA-alpha1.6-tamed-dt0.0005-20260806T222424452663Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/FLA/FLA-alpha1.6-tamed-dt0.0005-20260806T222424452663Z',
  'dt': 0.0005,
  'calibration_hash': '3e437feaedd98ae1c51e246f6dc69662',
  'fee_calibration_hash': '617414aeae69572deaa374dae2d362a1',
  'n_metric_rows': 884,
  'n_snapshots': 4},
 {'variant_label': 'FLA alpha=1.7, canonical',
  'status': 'complete',


## ULD

ULD is the method; BAOAB is the integrator it is discretised with. Runs, manifests, and legends say ULD.

In [6]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="ULD",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[ULD gamma=1, canonical] saved to /home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/ULD/ULD-gamma1-canonical-dt0.002-20260806T222808226190Z


[ULD gamma=1, tamed] saved to /home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/ULD/ULD-gamma1-tamed-dt0.002-20260806T222824503194Z


[{'variant_label': 'ULD gamma=1, canonical',
  'status': 'complete',
  'run_id': 'ULD-gamma1-canonical-dt0.002-20260806T222808226190Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/ULD/ULD-gamma1-canonical-dt0.002-20260806T222808226190Z',
  'dt': 0.002,
  'calibration_hash': '85916bba7bef25577215683e55b8c346',
  'fee_calibration_hash': '617414aeae69572deaa374dae2d362a1',
  'n_metric_rows': 884,
  'n_snapshots': 4},
 {'variant_label': 'ULD gamma=1, tamed',
  'status': 'complete',
  'run_id': 'ULD-gamma1-tamed-dt0.002-20260806T222824503194Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/ULD/ULD-gamma1-tamed-dt0.002-20260806T222824503194Z',
  'dt': 0.002,
  'calibration_hash': '1a2059b098b60d5abb86dd4f6aa5fdbc',
  'fee_calibration_hash': '617414aeae69572deaa374dae2d362a1',
  'n_metric_rows': 884,
  'n_snapshots': 4}]

## PT

Parallel tempering. The replica ladder is tuned by the calibration step that `run_variants_and_save` invokes, not here, and the tuned ladder is written into the run's `calibration.json`.

In [7]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="PT",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[PT n_swap=10, canonical] NOT CALIBRATABLE: unstable at every timestep tried (mh_acceptance_outside_target_band) and it does not improve as the timestep shrinks


[PT n_swap=10, tamed] saved to /home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/PT/PT-n_swap10-tamed-dt0.004-20260806T223440613200Z


[{'variant_label': 'PT n_swap=10, canonical',
  'method': 'PT',
  'status': 'uncalibratable',
  'diagnosis': 'unstable at every timestep tried (mh_acceptance_outside_target_band) and it does not improve as the timestep shrinks',
  'calibration_kind': 'timestep',
  'calibration_table': [{'dt': 0.004,
    'pass': False,
    'stability_problems': [('mh_acceptance_outside_target_band',
      0.7513866937419974)],
    'agreement_failures': [],
    'summary': {'n_steps': 1562,
     'nonfinite_fraction': 0.0,
     'boundary_reject_fraction': 0.0,
     'n_effective': 1024,
     'summary_mean': -0.9657288718637757,
     'summary_mean_se': 0.005352221294255742,
     'summary_abs_mean': 0.9710579738325948,
     'summary_abs_mean_se': 0.0044812371924876065,
     'summary_median': -0.9813284144656508,
     'summary_median_se': 0.005405280787606662,
     'summary_iqr': 0.1687779329409298,
     'summary_iqr_se': 0.006404205391334443,
     'energy_mean': 1.471748549647745,
     'energy_mean_se': 0.013

## Raw-CP

The same compound-Poisson jump process with the Levy score correction switched off. It does not preserve the target, so it is the control arm that isolates what the score correction buys, not a competitive baseline.

In [8]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="Raw-CP",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[Raw-CP, canonical] saved to /home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/Raw-CP/Raw-CP-canonical-dt0.002-20260806T223504629726Z


[Raw-CP, tamed] saved to /home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/Raw-CP/Raw-CP-tamed-dt0.002-20260806T223529842816Z


[{'variant_label': 'Raw-CP, canonical',
  'status': 'complete',
  'run_id': 'Raw-CP-canonical-dt0.002-20260806T223504629726Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/Raw-CP/Raw-CP-canonical-dt0.002-20260806T223504629726Z',
  'dt': 0.002,
  'calibration_hash': '413ff49b379b5011b778d87d73db1113',
  'fee_calibration_hash': '617414aeae69572deaa374dae2d362a1',
  'n_metric_rows': 884,
  'n_snapshots': 4},
 {'variant_label': 'Raw-CP, tamed',
  'status': 'complete',
  'run_id': 'Raw-CP-tamed-dt0.002-20260806T223529842816Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/Raw-CP/Raw-CP-tamed-dt0.002-20260806T223529842816Z',
  'dt': 0.002,
  'calibration_hash': '79f72591d89d7c0b0be33555c6589f23',
  'fee_calibration_hash': '617414aeae69572deaa374dae2d362a1',
  'n_metric_rows': 884,
  'n_snapshots': 4}]

## LSC-CP

Compound-Poisson jumps with the full deterministic-quadrature Levy score correction.

In [9]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="LSC-CP",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[LSC-CP, canonical] NOT CALIBRATABLE: unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks


[LSC-CP, tamed] saved to /home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/LSC-CP/LSC-CP-tamed-dt0.002-20260806T223910569248Z


[{'variant_label': 'LSC-CP, canonical',
  'method': 'LSC-CP',
  'status': 'uncalibratable',
  'diagnosis': 'unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks',
  'calibration_kind': 'timestep',
  'calibration_table': [{'dt': 0.002,
    'pass': False,
    'stability_problems': [('boundary_reject_fraction', 0.6916659375)],
    'agreement_failures': [],
    'summary': {'n_steps': 3125,
     'nonfinite_fraction': 0.0,
     'boundary_reject_fraction': 0.6916659375,
     'n_effective': 1024,
     'summary_mean': -1.0782188392588132,
     'summary_mean_se': 0.05990876446260207,
     'summary_abs_mean': 1.9096159780312525,
     'summary_abs_mean_se': 0.0327619408359387,
     'summary_median': -1.0105934530068135,
     'summary_median_se': 0.007970287698322863,
     'summary_iqr': 2.0901230209883557,
     'summary_iqr_se': 0.12310469971777731,
     'energy_mean': 60.16792572917427,
     'energy_mean_se': 0.5858708790041867,
     'energy_

## LSC-CP-RA

`A` is the **iid Monte Carlo bank size of one estimator family**, LSC-CP-RA. `A = 1, 4, 8` are variants of that single family, not three separate methods, and **all of them run from this one cell** and save as separate variants.

The bank holds `A` displacements drawn iid from the full normalised jump law `rho = nu / lambda`, and **the same bank drives both the score and the compound-Poisson increment**.

In [10]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="LSC-CP-RA",
    variants=[
        {"A": 1}, {"A": 4}, {"A": 8},
    ],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[LSC-CP-RA, canonical] NOT CALIBRATABLE: unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks


[LSC-CP-RA, tamed] saved to /home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/LSC-CP-RA/LSC-CP-RA-A1-tamed-dt0.002-20260806T224237850536Z


[LSC-CP-RA (A=4), canonical] NOT CALIBRATABLE: unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks


[LSC-CP-RA (A=4), tamed] saved to /home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/LSC-CP-RA/LSC-CP-RA-A4-tamed-dt0.002-20260806T224615121639Z


[LSC-CP-RA (A=8), canonical] NOT CALIBRATABLE: unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks


[LSC-CP-RA (A=8), tamed] saved to /home/zheyuanlai/levy-sampling/results/E4_coupled_quartic_chain/runs/LSC-CP-RA/LSC-CP-RA-A8-tamed-dt0.002-20260806T225013437634Z


[{'variant_label': 'LSC-CP-RA, canonical',
  'method': 'LSC-CP-RA',
  'status': 'uncalibratable',
  'diagnosis': 'unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks',
  'calibration_kind': 'timestep',
  'calibration_table': [{'dt': 0.002,
    'pass': False,
    'stability_problems': [('boundary_reject_fraction', 0.178183125)],
    'agreement_failures': [],
    'summary': {'n_steps': 3125,
     'nonfinite_fraction': 0.0,
     'boundary_reject_fraction': 0.178183125,
     'n_effective': 1024,
     'summary_mean': -0.01817590565183166,
     'summary_mean_se': 0.049650073906181355,
     'summary_abs_mean': 1.4203239192928883,
     'summary_abs_mean_se': 0.020470315224574737,
     'summary_median': -0.6236474289344702,
     'summary_median_se': 0.6269236502864897,
     'summary_iqr': 2.2826868840199603,
     'summary_iqr_se': 0.169991517033239,
     'energy_mean': 16.004643316996095,
     'energy_mean_se': 1.0942355009266436,
     'en

## Rebuild the catalog

`catalog.csv` is a **derived index** over the run manifests. It is never written by a worker mid-run, so concurrent runs never contend for it, and it can be rebuilt at any time by rescanning the manifests -- a lost or stale catalog costs nothing. Only runs that verify (manifest present, `COMPLETE` present, hashes matching) are admitted.

In [11]:
from src.catalog import write_catalog

report = write_catalog(experiment.paths.experiment_dir)
print(f"catalog rebuilt: {report['n_runs']} runs, {report['n_rejected']} rejected")

catalog rebuilt: 72 runs, 0 rejected
